# float

Floats are for approximation, not exact decimal truth. They matter in monitoring, latency analysis, percentages, and scientific workloads, but they become dangerous when engineers expect exact equality.


## 1. Problem First

Why does this matter in real systems?

- Latency averages are often floats.
- Billing code breaks if floats are used for money.
- Threshold alerts can flap because `0.1 + 0.2` is not exactly `0.3`. 


In [ ]:
response_times = [120.5, 98.2, 101.3]
average = sum(response_times) / len(response_times)
print(average)


## 2. Minimal Concept

Python `float` stores real-looking numbers approximately.

- `3.14`
- `0.5`
- `-2.0`
- Supports arithmetic with binary rounding underneath


## 3. Mental Model

How Python actually behaves

Python floats use binary floating-point representation. Many decimal fractions cannot be represented exactly in binary, so Python stores the nearest representable value.


In [ ]:
x = 0.1
y = 0.2
z = x + y
print(z)
print(z == 0.3)
print(repr(z))


## 4. Internal Mechanics

The exact stored approximation can be exposed with `.hex()` and compared safely with `math.isclose()` when approximation is acceptable.


In [ ]:
import dis
import math

value = 0.1 + 0.2
print(value.hex())
print(math.isclose(value, 0.3))

def add(a, b):
    return a + b

dis.dis(add)


## 5. Edge Cases

Where it breaks

- Equality checks on computed floats are fragile.
- Repeated addition accumulates error.
- `round()` is presentation logic, not mathematical truth.
- `NaN` is not equal to itself.


In [ ]:
total = 0.0
for _ in range(10):
    total += 0.1
print(total)
print(total == 1.0)

nan_value = float("nan")
print(nan_value == nan_value)


## 6. Performance Thinking

- Float operations are usually O(1).
- They are fast and compact for metrics and scientific work.
- They are the wrong tool for exact financial arithmetic.
- Approximation cost is usually correctness, not speed.


## 7. Real Use Case

A monitoring service computes rolling average latency and compares it against an alert threshold.


In [ ]:
import math

latencies_ms = [102.1, 108.4, 97.9, 110.0]
average_latency = sum(latencies_ms) / len(latencies_ms)
print(average_latency)
print(average_latency > 105.0)
print(math.isclose(average_latency, 105.0, rel_tol=1e-9))


## 8. Anti-Pattern

What beginners do wrong

- Use floats for money.
- Compare computed floats with `==`.
- Assume printed output reveals exact storage.


In [ ]:
price = 0.1 + 0.1 + 0.1
print(price)
print(price == 0.3)


## 9. Interview Signals

Questions FAANG asks

- Why does `0.1 + 0.2 != 0.3`?
- When should you use `Decimal` instead of `float`?
- What does `math.isclose()` solve?
- Why is `NaN != NaN`?


## 10. Exercise (Non-trivial)

Build alert logic on average CPU usage that avoids false positives caused by tiny floating-point differences. Explain when approximate comparison is acceptable.


In [ ]:
def should_alert(current_average, threshold):
    return current_average > threshold

# Task:
# 1. Decide whether strict or tolerant comparison is appropriate.
# 2. Justify the tolerance choice.
# 3. Explain why you would not use float for billing.
